In [0]:
# Verify UC functions exist – must specify the same catalog/schema used at registration
from dotenv import load_dotenv
load_dotenv()  # ensure credentials are available

from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient(catalog="main", schema="default")
funcs = ["main.default.compound_growth", "main.default.percent_change", "main.default.calculate"]
for f in funcs:
    try:
        info = client.get_function(f)
        print(f"✅ {f}")
    except Exception as e:
        print(f"❌ {f}: {e}")

In [0]:
# Build the graph that uses UC functions and ask a calculation question
from agent.graph_uc import build_graph
from langchain_core.messages import HumanMessage

graph = build_graph()
query = "What is 15% of 2.4 billion?"
result = graph.invoke({"messages": [HumanMessage(content=query)]})

print("📌 Plan:", result.get("plan"))
print("\n⚙️ Step results:")
for i, res in enumerate(result.get("step_results", []), 1):
    print(f"   Step {i}: {res}")

final = result["messages"][-1].content if result.get("messages") else "⚠️ No messages"
print(f"\n✅ Final answer:\n{final}")

# PART 2

In [0]:
%pip install databricks-langchain unitycatalog-ai unitycatalog-langchain langgraph langchain langchain-openai databricks-sdk

In [0]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # one level up from genie/
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repo root added:", repo_root)

In [0]:
# Create meridian_financials table
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

data = [
    Row(year=2019, net_revenue_billion=11.28, operating_profit_billion=0.89, net_income_billion=0.55),
    Row(year=2020, net_revenue_billion=12.50, operating_profit_billion=0.92, net_income_billion=0.58),
    Row(year=2021, net_revenue_billion=13.80, operating_profit_billion=0.98, net_income_billion=0.62),
    Row(year=2022, net_revenue_billion=14.55, operating_profit_billion=0.905, net_income_billion=0.68),
    Row(year=2023, net_revenue_billion=16.91, operating_profit_billion=1.124, net_income_billion=0.75),
]

schema = StructType([
    StructField("year", IntegerType(), False),
    StructField("net_revenue_billion", DoubleType(), False),
    StructField("operating_profit_billion", DoubleType(), False),
    StructField("net_income_billion", DoubleType(), False),
])

df = spark.createDataFrame(data, schema=schema)
df.write.mode("overwrite").saveAsTable("main.default.meridian_financials")
print("Table main.default.meridian_financials created.")

In [0]:
%pip install -U typing_extensions

In [0]:
import os

# ---- Set the same values as in your .env ----
os.environ["DATABRICKS_HOST"] = ""   # your workspace URL
os.environ["DATABRICKS_TOKEN"] = ""                     # your personal access token (or notebook token)
os.environ["DATABRICKS_MODEL"] = "databricks-meta-llama-3-3-70b-instruct"
os.environ["EMBEDDINGS_ENDPOINT"] = "databricks-gte-large-en"
os.environ["UC_CATALOG"] = "main"
os.environ["UC_SCHEMA"] = "default"
os.environ["VECTOR_SEARCH_ENDPOINT"] = "mehmood-vs-endpoint"
os.environ["VECTOR_SEARCH_INDEX"] = "main.default.mehmood_analyst_index"
# If using a SQL warehouse, also set:
# os.environ["SQL_WAREHOUSE_ID"] = "your-warehouse-id"

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState
from config import get_chat_llm
import re

SQL_WAREHOUSE_ID = "729a11c08de709df"

def ask_genie(question: str) -> str:
    # 1. Generate SQL using LLM
    llm = get_chat_llm()
    prompt = f"""You are a SQL expert. Given a table `main.default.meridian_financials`
with columns: year (int), net_revenue_billion (double), operating_profit_billion (double),
net_income_billion (double). Write a SQL query that answers the following question.
Return only the SQL query, no explanation.
Question: {question}
SQL:"""
    response = llm.invoke(prompt)
    raw = response.content.strip()
    
    # Strip markdown code fences if present
    match = re.search(r"```sql\s*(.*?)\s*```", raw, re.DOTALL | re.IGNORECASE)
    sql = match.group(1).strip() if match else raw
    print(f"Generated SQL:\n{sql}")
    
    # 2. Execute SQL on the warehouse
    w = WorkspaceClient()
    stmt = w.statement_execution.execute_statement(
        warehouse_id=SQL_WAREHOUSE_ID,
        catalog="main",
        schema="default",
        statement=sql,
        wait_timeout="30s"
    )
    if stmt.status.state == StatementState.SUCCEEDED:
        rows = stmt.result.data_array
        if rows:
            return f"Query result: {rows}"
        return "Query returned no rows."
    # Access the error message correctly
    error_msg = getattr(stmt.status, 'message', None) or getattr(stmt.status, 'error_message', None) or str(stmt.status)
    return f"SQL execution failed: {error_msg}"

# Test the client
print(ask_genie("What was the net income in 2021?"))

In [0]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))   # up from genie/ to pa4/
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repo root:", repo_root)

In [0]:
%pip install langgraph langchain langchain-openai databricks-langchain unitycatalog-ai unitycatalog-langchain

In [0]:
%pip install databricks-vectorsearch

In [0]:
import re

def make_mcp_node(tools, llm):
    """Node that executes a calculation step, substituting previous results."""
    llm_with_tools = llm.bind_tools(tools)

    MCP_SYSTEM_PROMPT = """\
You are a calculation assistant with access to precise math tools.
For the given step, choose exactly one tool and call it with the correct arguments.\
"""

    def _extract_number_from_results(term: str, previous_results: list[str]) -> float | None:
        """Search previous results for a number associated with a term like 'revenue'."""
        # Simple heuristic: look for a pattern like ¥16.91 trillion, 16.91 billion, etc.
        for res in reversed(previous_results):
            match = re.search(r"(\d+\.?\d*)\s*(trillion|billion|million|thousand)?", res.lower())
            if match:
                value = float(match.group(1))
                unit = match.group(2)
                if unit == "trillion":
                    value *= 1e12
                elif unit == "billion":
                    value *= 1e9
                elif unit == "million":
                    value *= 1e6
                elif unit == "thousand":
                    value *= 1e3
                return value
        return None

    def _substitute_in_step(step: str, previous_results: list[str]) -> str:
        """Replace known financial terms with numeric values from previous results."""
        terms = ["revenue", "income", "profit"]
        for term in terms:
            if term in step.lower():
                value = _extract_number_from_results(term, previous_results)
                if value is not None:
                    # Replace the term AND remove surrounding words like "in 2023"
                    # Example: "revenue in 2023" -> "16910000000000"
                    step = re.sub(
                        rf"\b{term}\b\s*(in\s+\d{{4}})?",
                        str(value),
                        step,
                        flags=re.IGNORECASE,
                    )
        return step

    def mcp_tools(state: dict) -> dict:
        plan = state.get("plan", [])
        idx = state.get("current_step_index", 0)
        if idx >= len(plan):
            return state

        step = plan[idx]
        previous_results = state.get("step_results", [])

        # Substitute variables in the step text itself before showing it to the LLM
        enriched_step = _substitute_in_step(step, previous_results)

        response = llm_with_tools.invoke([
            {"role": "system", "content": MCP_SYSTEM_PROMPT},
            {"role": "user", "content": enriched_step},
        ])

        if not response.tool_calls:
            result = f"Unable to process step: {step}"
        else:
            tool_call = response.tool_calls[0]
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            # Second layer of cleaning: ensure no remaining words in arguments
            cleaned_args = {}
            for arg_name, arg_value in tool_args.items():
                if isinstance(arg_value, str):
                    cleaned_args[arg_name] = _substitute_in_step(
                        arg_value, previous_results
                    )
                else:
                    cleaned_args[arg_name] = arg_value

            tool_obj = next((t for t in tools if t.name == tool_name), None)
            if tool_obj is None:
                result = f"Tool '{tool_name}' not found."
            else:
                try:
                    result = str(tool_obj.invoke(cleaned_args))
                except Exception as e:
                    result = f"Tool error: {e}"

        new_results = state.get("step_results", []) + [result]
        return {
            "step_results": new_results,
            "current_step_index": idx + 1,
        }

    return mcp_tools

print("✅ Improved make_mcp_node (with step‑level substitution) defined")

In [0]:
# from agent.graph_uc import make_mcp_node

In [0]:
from agent.planner import make_planner
from agent.rag_agent import make_rag_agent
from agent.supervisor import MCP, RAG, SYNTH, make_supervisor, route_from_supervisor
from agent.synthesizer import make_synthesizer
from config import get_chat_llm
from rag.store import get_retriever
from databricks_langchain import UCFunctionToolkit
from langgraph.graph import END, START, StateGraph
from agent.state import AnalystState
from langchain_core.messages import HumanMessage

# ----- UC tools (from Part 1) -----
toolkit = UCFunctionToolkit(
    function_names=["main.default.compound_growth", "main.default.percent_change", "main.default.calculate"]
)
tools = toolkit.tools

# ----- Define genie_node using the ask_genie we defined above -----
def genie_node(state: dict) -> dict:
    plan = state.get("plan", [])
    idx = state.get("current_step_index", 0)
    if idx >= len(plan):
        return state
    step = plan[idx]
    answer = ask_genie(step)          # uses the function from the earlier cell
    result_text = f"From Genie: {answer}"
    new_results = state.get("step_results", []) + [result_text]
    return {
        "step_results": new_results,
        "current_step_index": idx + 1,
    }

# ----- Extended supervisor (routes tabular keywords to genie_agent) -----
def extended_supervisor(state: dict) -> dict:
    plan = state.get("plan", [])
    idx = state.get("current_step_index", 0)
    if idx >= len(plan):
        return {"next_agent": "synthesizer"}
    step = plan[idx]
    lower = step.lower()
    if lower.startswith("retrieve:"):
        # If the step contains tabular keywords, route to Genie
        if any(w in lower for w in ["year", "table", "trend", "compare", "highest", "lowest", "sql", "tabular"]):
            return {"next_agent": "genie_agent"}
        return {"next_agent": "rag_agent"}
    elif lower.startswith("compute:"):
        return {"next_agent": "mcp_tools"}
    # Fallback
    return {"next_agent": "rag_agent"}

def route_from_supervisor(state):
    return state.get("next_agent", "synthesizer")

# ----- Assemble the graph -----
llm = get_chat_llm()
retriever = get_retriever()

planner_node = make_planner(llm)
supervisor_node = extended_supervisor
rag_node = make_rag_agent(retriever, llm)
mcp_node = make_mcp_node(tools, llm)      # from agent/graph_uc (UC‑powered)
synth_node = make_synthesizer(llm)

builder = StateGraph(AnalystState)
builder.add_node("planner", planner_node)
builder.add_node("supervisor", supervisor_node)
builder.add_node("rag_agent", rag_node)
builder.add_node("mcp_tools", mcp_node)
builder.add_node("genie_agent", genie_node)
builder.add_node("synthesizer", synth_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "supervisor")
builder.add_conditional_edges("supervisor", route_from_supervisor, {
    "rag_agent": "rag_agent",
    "mcp_tools": "mcp_tools",
    "genie_agent": "genie_agent",
    "synthesizer": "synthesizer",
})
builder.add_edge("rag_agent", "supervisor")
builder.add_edge("mcp_tools", "supervisor")
builder.add_edge("genie_agent", "supervisor")
builder.add_edge("synthesizer", END)

graph = builder.compile()
print("✅ Multi‑modal graph built.")

In [0]:
from langchain_core.messages import HumanMessage

query = "What was the net income in 2021?"
result = graph.invoke({"messages": [HumanMessage(content=query)]})

print("📌 Plan:", result.get("plan"))
print("\n⚙️ Step results:")
for i, res in enumerate(result.get("step_results", []), 1):
    print(f"   Step {i}: {res}")

final = result["messages"][-1].content if result.get("messages") else "⚠️ No messages"
print(f"\n✅ Final answer:\n{final}")

In [0]:
query2 = "What was the revenue in 2023, and what would a 10% increase look like?"
result2 = graph.invoke({"messages": [HumanMessage(content=query2)]})

print("📌 Plan:", result2.get("plan"))
print("\n⚙️ Step results:")
for i, res in enumerate(result2.get("step_results", []), 1):
    print(f"   Step {i}: {res}")

final2 = result2["messages"][-1].content if result2.get("messages") else "⚠️ No messages"
print(f"\n✅ Final answer:\n{final2}")

#Task 3 

In [0]:
# Baseline make_mcp_node (no substitution) – used for "before" evaluation
def make_mcp_node_baseline(tools, llm):
    llm_with_tools = llm.bind_tools(tools)
    MCP_SYSTEM_PROMPT = """\
You are a calculation assistant with access to precise math tools.
For the given step, choose exactly one tool and call it with the correct arguments.\
"""
    def mcp_tools(state: dict) -> dict:
        plan = state.get("plan", [])
        idx = state.get("current_step_index", 0)
        if idx >= len(plan):
            return state
        step = plan[idx]
        response = llm_with_tools.invoke([
            {"role": "system", "content": MCP_SYSTEM_PROMPT},
            {"role": "user", "content": step},
        ])
        if not response.tool_calls:
            result = f"Unable to process step: {step}"
        else:
            tool_call = response.tool_calls[0]
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_obj = next((t for t in tools if t.name == tool_name), None)
            if tool_obj is None:
                result = f"Tool '{tool_name}' not found."
            else:
                try:
                    result = str(tool_obj.invoke(tool_args))
                except Exception as e:
                    result = f"Tool error: {e}"
        new_results = state.get("step_results", []) + [result]
        return {"step_results": new_results, "current_step_index": idx + 1}
    return mcp_tools

In [0]:
import json
from langchain_core.messages import HumanMessage

# Load dataset (adjust path to your uploaded file)
with open("/Workspace/Users/28100008@lums.edu.pk/cs4603-pa4/eval/eval_dataset.jsonl") as f:
    eval_data = [json.loads(line) for line in f]

def evaluate_graph(graph, dataset):
    scores = []
    for item in dataset:
        question = item["question"]
        expected = item["expected_answer"].lower()
        result = graph.invoke({"messages": [HumanMessage(content=question)]})
        answer = result["messages"][-1].content.lower()
        # Simple metric: does expected answer appear in the generated answer?
        correct = expected in answer
        scores.append(correct)
        print(f"Q: {question}\nExpected: {expected}\nGot: {answer}\nCorrect: {correct}\n")
    return sum(scores) / len(scores)  # accuracy

In [0]:
from agent.planner import make_planner
from agent.rag_agent import make_rag_agent
from agent.synthesizer import make_synthesizer
from agent.state import AnalystState
from config import get_chat_llm
from rag.store import get_retriever
from databricks_langchain import UCFunctionToolkit
from langgraph.graph import END, START, StateGraph

llm = get_chat_llm()
retriever = get_retriever()
toolkit = UCFunctionToolkit(function_names=["main.default.compound_growth", "main.default.percent_change", "main.default.calculate"])
tools = toolkit.tools

# Baseline node (no substitution)
mcp_node_baseline = make_mcp_node_baseline(tools, llm)

planner_node = make_planner(llm)
# Use the existing extended_supervisor (already defined in notebook) or redefine it here
def extended_supervisor(state: dict) -> dict:
    plan = state.get("plan", [])
    idx = state.get("current_step_index", 0)
    if idx >= len(plan):
        return {"next_agent": "synthesizer"}
    step = plan[idx]
    lower = step.lower()
    if lower.startswith("retrieve:"):
        return {"next_agent": "rag_agent"}
    elif lower.startswith("compute:"):
        return {"next_agent": "mcp_tools"}
    return {"next_agent": "rag_agent"}

def route_from_supervisor(state):
    return state.get("next_agent", "synthesizer")

supervisor_node = extended_supervisor
rag_node = make_rag_agent(retriever, llm)
synth_node = make_synthesizer(llm)

builder = StateGraph(AnalystState)
builder.add_node("planner", planner_node)
builder.add_node("supervisor", supervisor_node)
builder.add_node("rag_agent", rag_node)
builder.add_node("mcp_tools", mcp_node_baseline)
builder.add_node("synthesizer", synth_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "supervisor")
# Note: no genie_agent in this evaluation graph
builder.add_conditional_edges("supervisor", route_from_supervisor, {
    "rag_agent": "rag_agent",
    "mcp_tools": "mcp_tools",
    "synthesizer": "synthesizer",
})
builder.add_edge("rag_agent", "supervisor")
builder.add_edge("mcp_tools", "supervisor")
builder.add_edge("synthesizer", END)

graph_baseline = builder.compile()
print("✅ Baseline graph built")

baseline_accuracy = evaluate_graph(graph_baseline, eval_data)
print(f"Baseline accuracy: {baseline_accuracy:.2%}")

In [0]:
# Fixed graph (with substitution) — no genie_agent in this evaluation graph
mcp_node_fixed = make_mcp_node(tools, llm)

builder2 = StateGraph(AnalystState)
builder2.add_node("planner", planner_node)
builder2.add_node("supervisor", supervisor_node)
builder2.add_node("rag_agent", rag_node)
builder2.add_node("mcp_tools", mcp_node_fixed)
builder2.add_node("synthesizer", synth_node)

builder2.add_edge(START, "planner")
builder2.add_edge("planner", "supervisor")
builder2.add_conditional_edges("supervisor", route_from_supervisor, {
    "rag_agent": "rag_agent",
    "mcp_tools": "mcp_tools",
    "synthesizer": "synthesizer",
})   # ← removed genie_agent
builder2.add_edge("rag_agent", "supervisor")
builder2.add_edge("mcp_tools", "supervisor")
builder2.add_edge("synthesizer", END)

graph_fixed = builder2.compile()
print("✅ Fixed graph built")

fixed_accuracy = evaluate_graph(graph_fixed, eval_data)
print(f"Fixed accuracy: {fixed_accuracy:.2%}")

# PART 4 : A

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
try:
    ep = w.serving_endpoints.get("pa4-doc-analyst-endpoint")
    print("Endpoint state:", ep.state.ready)
except Exception as e:
    print("Endpoint not found:", e)

In [0]:
import mlflow

# Enable MLflow LangChain auto-tracing (no extra arguments needed)
mlflow.langchain.autolog()

# Use your existing multi‑modal graph
from agent.graph_multi import build_graph
from langchain_core.messages import HumanMessage

graph = build_graph()

# Run a query inside a traced run
with mlflow.start_run(run_name="part4-trace-demo") as run:
    result = graph.invoke({"messages": [HumanMessage(content="What was the net income in 2021?")]})
    print("🔍 Traced run ID:", run.info.run_id)
    print("✅ Answer:", result["messages"][-1].content)

In [0]:
# %sql
-- Create an inference table for the endpoint
CREATE INFERENCE TABLE my_endpoint_logs
FOR ENDPOINT pa4-doc-analyst-endpoint
USING DELTA
LOCATION 'dbfs:/user/hive/warehouse/my_endpoint_logs';

# Part 4 : E

In [0]:
from bonus.guardrails import validate_answer, safe_fallback_answer
from agent.synthesizer import make_synthesizer
from langchain_core.messages import AIMessage

original_make_synthesizer = make_synthesizer

def make_synthesizer_with_guardrail(llm):
    base_synth = original_make_synthesizer(llm)
    def guarded_synthesizer(state: dict) -> dict:
        result = base_synth(state)
        answer = result.get("final_answer", "")
        question = ""
        for msg in state.get("messages", []):
            if hasattr(msg, "type") and msg.type == "human":
                question = msg.content
                break
        if validate_answer(question, answer):
            return result
        else:
            fallback = safe_fallback_answer()
            return {
                "final_answer": fallback,
                "messages": [AIMessage(content=fallback)],
            }
    return guarded_synthesizer

# Rebuild graph with guarded synthesizer (use the same structure as before but replace synth)

In [0]:
# ---------- Quick verification of the guardrail ----------
from unittest.mock import MagicMock 
from langchain_core.messages import HumanMessage, AIMessage

# Create a dummy LLM that returns a purposely bad answer (no number)
dummy_llm = MagicMock()
dummy_llm.invoke.return_value = AIMessage(content="I'm not sure about the revenue.")

# Build the guarded synthesizer with this dummy LLM
guarded_synth = make_synthesizer_with_guardrail(dummy_llm)

# Simulate a state where the user asked a numeric question
test_state = {
    "messages": [HumanMessage(content="What was the revenue in 2023?")],
    "step_results": ["Some retrieval result"],
    "plan": [],
    "current_step_index": 0,
    "next_agent": "synthesizer",
}

# Call the synthesizer
result = guarded_synth(test_state)

# Check the output
print("Final answer from guardrail:")
print(result["final_answer"])
print()
if "couldn't provide a valid answer" in result["final_answer"]:
    print("✅ Guardrail correctly replaced bad answer with fallback.")
else:
    print("❌ Guardrail did NOT trigger as expected.")

# PART 4 : F

In [0]:
import mlflow

planner_text = """You are a planning assistant. Break down the user's question into a list of atomic steps (2–5). Each step should begin with either "Retrieve:" (for facts found in the document) or "Compute:" (for math/calculations). Return ONLY a JSON array of strings."""

prompt_name = "plannerprompt"   # no underscores, spaces, or special characters

# 1. Register the prompt
try:
    mlflow.genai.register_prompt(
        name=prompt_name,
        template=planner_text,
        tags={"task": "planning", "agent": "document-analyst"}
    )
    print("✅ Prompt registered.")
except Exception as e:
    print("⚠️ Prompt registration failed:", e)

# 2. Set production alias
try:
    mlflow.genai.set_prompt_alias(prompt_name, "production", version=1)
    print("✅ Promoted version 1 to production alias.")
except Exception as e:
    print("⚠️ Could not set alias:", e)

# 3. Load and display
try:
    prod_prompt = mlflow.genai.load_prompt(f"{prompt_name}@production")
    print("✅ Production prompt loaded. First 100 chars:", prod_prompt.template[:100])
except Exception as e:
    print("⚠️ Could not load prompt:", e)